phase 4.2

In [143]:
import pandas as pd
import numpy as np
import os

from sentence_transformers import SentenceTransformer
import chromadb

df = pd.read_csv('../data/processed/cleaned_medical_notes.csv')
print(df.shape)
print(df.columns.tolist())   # confirm 'transcription' and 'medical_specialty' are here

(4951, 9)
['description', 'medical_specialty', 'sample_name', 'transcription', 'keywords', 'char_lenght', 'word_count', 'sentence_count', 'clean_text']


In [144]:
not_specialties = [
    'SOAP / Chart / Progress Notes', 'Office Notes', 'Letters',
    'Discharge Summary', 'Emergency Room Reports',
    'Consult - History and Phy.', 'IME-QME-Work Comp etc.'
]

df = df[~df['medical_specialty'].isin(not_specialties)].reset_index(drop=True)
print("rows after filtering:", len(df))
print("specialties remaining:", df['medical_specialty'].nunique())

rows after filtering: 4001
specialties remaining: 33


In [145]:
def chunk_text(text, chunk_size=200, overlap=20):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        chunk = words[start : start + chunk_size]
        if len(chunk) == 0:
            break
        chunks.append(' '.join(chunk))
    return chunks

In [146]:
chunk_texts = []
chunk_specialties = []
chunk_note_ids = []

for note_id, row in df.iterrows():
    pieces = chunk_text(row['transcription'])
    for piece in pieces:
        chunk_texts.append(piece)
        chunk_specialties.append(row['medical_specialty'])
        chunk_note_ids.append(note_id)

print(f"total notes: {len(df)}")
print(f"total chunks: {len(chunk_texts)}")

total notes: 4001
total chunks: 12210


In [147]:
emb_path = '../data/processed/embeddings.npy'

if os.path.exists(emb_path):
    embeddings = np.load(emb_path)          # instant — load cached
    print("loaded cached embeddings:", embeddings.shape)
else:
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(chunk_texts, show_progress_bar=True, batch_size=64)
    np.save(emb_path, embeddings)           # save for next time
    print("computed and saved:", embeddings.shape)

loaded cached embeddings: (12210, 384)


In [148]:
client = chromadb.PersistentClient(path='../data/processed/chroma_db')

existing = [c.name for c in client.list_collections()]

if "medical_notes" in existing:
    # already built and saved — just reconnect (instant)
    collection = client.get_collection("medical_notes")
    print("reconnected to existing DB, count:", collection.count())
else:
    # first time only — build it
    collection = client.create_collection("medical_notes")
    all_embeddings = embeddings.tolist()
    all_metadatas  = [{"specialty": str(s), "note_id": int(n)}
                      for s, n in zip(chunk_specialties, chunk_note_ids)]
    all_ids        = [str(i) for i in range(len(chunk_texts))]
    BATCH = 5000
    for start in range(0, len(chunk_texts), BATCH):
        end = start + BATCH
        collection.add(
            embeddings=all_embeddings[start:end],
            documents=chunk_texts[start:end],
            metadatas=all_metadatas[start:end],
            ids=all_ids[start:end]
        )
        print(f"added {start} to {min(end, len(chunk_texts))}")
    print("built new DB, count:", collection.count())

reconnected to existing DB, count: 12210


pHASE 4.3 Mini Rag Classifier

In [149]:
from collections import Counter

def classify_by_retrieval(query, k=5):
    # 1. embed the query with the SAME model (lands it in the same 384-D space)
    query_vector = model.encode([query]).tolist()

    # 2. search the store for the k most similar chunks
    results = collection.query(
        query_embeddings=query_vector,
        n_results=k
    )

    # 3. pull the specialties of those k chunks and vote
    retrieved_specialties = [m['specialty'] for m in results['metadatas'][0]]
    vote = Counter(retrieved_specialties).most_common(1)[0][0]

    return vote, retrieved_specialties

In [150]:
pred, votes = classify_by_retrieval("Patient has persistent cough and wheezing")
print("Retrieved specialties:", votes)
print("Predicted:", pred)

# synonym test: "renal" never appears, but should retrieve kidney notes
pred, votes = classify_by_retrieval("the patient's kidneys are not filtering properly")
print("Query about kidney filtering →", pred, votes)

Retrieved specialties: ['Pediatrics - Neonatal', 'General Medicine', 'General Medicine', 'Cardiovascular / Pulmonary', 'General Medicine']
Predicted: General Medicine
Query about kidney filtering → Nephrology ['Nephrology', 'Urology', 'Nephrology', 'Nephrology', 'General Medicine']
